# NFL player trajectory · start here

**An executed research record, not a leaderboard claim.**

Read **01 · Data analysis** and **02 · Motion benchmarks** for the evidence and conclusions.
The completed feature experiment selected `landing_ridge`. Model generation is an explicit
user action at the end of notebook 02; reading or automatically publishing this repository
never submits anything to Kaggle.

The local raw data, frozen game splits, completed weekly caches, models, and private backups
remain separate from this public review. A missing local dataset is not a missing published result.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from nfl_trajectory.analysis import load_evidence
from nfl_trajectory.runtime import Run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/nfl_trajectory").is_dir())
with Run(ROOT, "notebook_readiness") as run:
    evidence, selection_audit, evidence_label = load_evidence(ROOT)
    run.event("evidence_verified", source=evidence_label, selected_model=evidence["selected_model"])
display(Markdown(f"**Evidence:** {evidence_label}. **Python:** {sys.version.split()[0]}."))

In [ ]:
checks = {
    "Published experiment": (ROOT / "docs/results/feature_summary.json").is_file(),
    "Local feature results": (ROOT / "artifacts/features/summary.json").is_file(),
    "Local fitted feature models": (ROOT / "artifacts/features/model.json").is_file(),
    "Local frozen split manifest": (ROOT / "artifacts/game_splits.csv").is_file(),
    "Local training data": (ROOT / "data/raw/train").is_dir(),
}
display(pd.DataFrame(checks.items(), columns=["Evidence or local capability", "Available"]))
display(Markdown(
    "Missing local artifacts do not invalidate the published experiment. "
    "They are required to resume training or create your own inference notebook."
))

## Explicit run/resume control

The feature run represented in these notebooks is already complete. Leave this control off
for reading, rendering, or reviewing the portfolio. Set it to `True` only to explicitly run
or resume the same experiment in your existing workspace. Verified checkpoints are reused;
this is not a command to recreate data or retrain the original baseline.

The S3 flag uses your existing private configuration; it does not create a new compute instance.
Normal AWS storage and existing-workspace charges may still apply. Automated notebook publication
ignores this opt-in control.

In [ ]:
RUN_FEATURE_EXPERIMENT = False
if RUN_FEATURE_EXPERIMENT and os.getenv("NFL_NOTEBOOK_AUTORUN") != "1":
    with Run(ROOT, "notebook_feature_experiment") as run:
        subprocess.run(
            [sys.executable, "-m", "nfl_trajectory.cli", "features", "--checkpoint-s3"],
            cwd=ROOT, check=True,
        )
        run.event("feature_command_completed")
else:
    display(Markdown("**Training not requested.** Completed measurements and checkpoints are preserved."))